# Post-processing: BinWaves + BMUS — 500 m reference points only

Crops per-grid **BinWaves + KMA-BMU** bulk hindcasts to the 500 m GeoJSON point list,
then smooth-merges all grids (8 variables only).

**Inputs:** `gridN/outputs/BinWaves_BMUS/{var}_gridN_BinWaves_BMUS.nc`  
**Outputs:**
- `outputs/cropped_500m_binwaves_bmus/{var}_gridN_points500m.nc`
- `outputs/merged_500m_binwaves_bmus/{var}_500m.nc`
- `webpage_binwaves_bmus_500m/wave_statistics_{all,hs,dp}.geojson`

Build per-grid BinWaves+BMUS if missing: `bash gridN/utils/build_kma_merged_grids.sh`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel")
GRID_NAMES = ["grid1", "grid2", "grid3", "grid4"]
POINTS_GEOJSON = PROJECT_ROOT / "inputs/water_level_statistics.geojson"

CROPPED_DIR = PROJECT_ROOT / "outputs/cropped_500m_binwaves_bmus"
MERGED_DIR = PROJECT_ROOT / "outputs/merged_500m_binwaves_bmus"
WEBPAGE_DIR = PROJECT_ROOT / "webpage_binwaves_bmus_500m"

VARIABLES = ["hs", "tp", "dm", "dp", "tm02", "phs0", "ptp0", "dp0"]

SMOOTH_STEEPNESS = 2.0
BLEND_BUFFER_KM = 30.0
TOLERANCE_DEG = 0.001
SKIP_EXISTING = True  # set False to rebuild

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import postprocessing_binwaves_bmus as pb

for d in (CROPPED_DIR, MERGED_DIR, WEBPAGE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Cropped output:", CROPPED_DIR)
print("Merged output:", MERGED_DIR)
print("Webpage output:", WEBPAGE_DIR)
print("Variables:", VARIABLES)

Project: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel
Cropped output: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/outputs/cropped_500m_binwaves_bmus
Merged output: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/outputs/merged_500m_binwaves_bmus
Webpage output: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/webpage_binwaves_bmus_500m
Variables: ['hs', 'tp', 'dm', 'dp', 'tm02', 'phs0', 'ptp0', 'dp0']


## 1. Audit per-grid BinWaves+BMUS inputs

In [ ]:
audit = pb.audit_grid_binwaves_bmus(PROJECT_ROOT, GRID_NAMES)
pivot = audit.pivot(index="variable", columns="grid", values="exists")
display(pivot.loc[VARIABLES])

missing = audit[(~audit["exists"]) & (audit["variable"].isin(VARIABLES))]
if not missing.empty:
    print("Missing files (run build_kma_merged_grids.sh per grid):")
    display(missing[["grid", "variable", "path"]])
else:
    print("All 8 variables present on all grids.")

## 2. Crop to 500 m reference points

Keeps only sites listed in `inputs/water_level_statistics.geojson`.

In [ ]:
cropped_paths = pb.crop_binwaves_bmus_to_points_geojson(
    project_root=PROJECT_ROOT,
    output_dir=CROPPED_DIR,
    points_geojson_file=POINTS_GEOJSON,
    grid_names=GRID_NAMES,
    variables=VARIABLES,
    skip_existing=SKIP_EXISTING,
    verbose=True,
)

print(f"\nCropped files: {len(cropped_paths)}")
for p in sorted(cropped_paths):
    size_mb = p.stat().st_size / 1e6 if p.is_file() else 0
    print(f"  {p.name} ({size_mb:.1f} MB)")

## 3. Smooth merge all grids (500 m)

All grids merged at once with buffer blending:
- **Directions** (`dp`, `dm`, `dp0`) — circular weighted mean
- **Scalars** (`hs`, `tp`, `tm02`, `phs0`, `ptp0`) — linear weighted mean
- Lower `SMOOTH_STEEPNESS` = wider overlap transition

In [2]:
# merged_paths = pb.merge_all_binwaves_bmus_smooth_from_cropped(
#     cropped_dir=CROPPED_DIR,
#     output_dir=MERGED_DIR,
#     grid_names=GRID_NAMES,
#     variables=VARIABLES,
#     steepness=SMOOTH_STEEPNESS,
#     blend_buffer_km=BLEND_BUFFER_KM,
#     tolerance_deg=TOLERANCE_DEG,
#     skip_existing=SKIP_EXISTING,
#     output_suffix="_500m",
# )

# print("Merged files:")
# for var, path in merged_paths.items():
#     size_mb = path.stat().st_size / 1e6 if path.is_file() else 0
#     blend = "circular" if var in pb.CIRCULAR_BLEND_VARS else "linear"
#     print(f"  {var:8s} {path.name}  ({size_mb:.1f} MB, {blend} blend)")

from pathlib import Path

CROPPED_DIR = Path("/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/outputs/cropped_500m_binwaves_bmus")
MERGED_DIR  = Path("/nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

merged_paths = pb.merge_all_binwaves_bmus_smooth_from_cropped(
    cropped_dir=CROPPED_DIR,   # read from old location
    output_dir=MERGED_DIR,     # write to new location
    grid_names=GRID_NAMES,
    variables=VARIABLES,
    steepness=SMOOTH_STEEPNESS,
    blend_buffer_km=BLEND_BUFFER_KM,
    tolerance_deg=TOLERANCE_DEG,
    skip_existing=SKIP_EXISTING,
    output_suffix="_500m",
)

Skip hs: hs_500m.nc exists
Skip tp: tp_500m.nc exists
Skip dm: dm_500m.nc exists
Smooth merge dp (circular) from 4 grid(s)...


/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dat

  dp: smooth merge:   0%|          | 0/1269 [00:00<?, ?it/s]

Saved /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/dp_500m.nc
  -> /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/dp_500m.nc
Smooth merge tm02 (linear) from 4 grid(s)...


/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dat

...

  tp: smooth merge:   0%|          | 0/1269 [00:00<?, ?it/s]

Saved /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/ptp0_500m.nc
  -> /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/ptp0_500m.nc
Smooth merge dp0 (circular) from 4 grid(s)...


/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dat

  dp: smooth merge:   0%|          | 0/1269 [00:00<?, ?it/s]

Saved /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/dp0_500m.nc
  -> /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/dp0_500m.nc


In [4]:
# 3b) Smooth merge partition variables (1,2,3) from ShoreShop cropped files
from pathlib import Path
import re

SHORESHOP_CROPPED_DIR = Path("/lustre/geocean/WORK/users/montanoj/personal/ShoreShop2026/outputs/cropped_500m")
MERGED_DIR = Path("/nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

# Keep only partition vars 1,2,3 (e.g., phs1/2/3, ptp1/2/3, dp1/2/3, spr1/2/3)
all_vars = pb.discover_cropped_500m_variables(SHORESHOP_CROPPED_DIR, GRID_NAMES)
partition_vars = [
    v for v in all_vars
    if re.match(r"^(phs|ptp|dp|spr)[123]$", v)
]

print("Partition variables to merge:", partition_vars)

merged_partition_paths = pb.merge_all_binwaves_bmus_smooth_from_cropped(
    cropped_dir=SHORESHOP_CROPPED_DIR,
    output_dir=MERGED_DIR,
    grid_names=GRID_NAMES,
    variables=partition_vars,
    steepness=SMOOTH_STEEPNESS,
    blend_buffer_km=BLEND_BUFFER_KM,
    tolerance_deg=TOLERANCE_DEG,
    skip_existing=SKIP_EXISTING,
    output_suffix="_500m",
)

print("Merged partition files:")
for var, path in merged_partition_paths.items():
    size_mb = path.stat().st_size / 1e6 if path.is_file() else 0
    blend = "circular" if var in pb.CIRCULAR_BLEND_VARS else "linear"
    print(f"  {var:8s} {path.name}  ({size_mb:.1f} MB, {blend} blend)")

Partition variables to merge: ['dp1', 'dp2', 'dp3', 'phs1', 'phs2', 'phs3', 'ptp1', 'ptp2', 'ptp3', 'spr1', 'spr2', 'spr3']
Smooth merge dp1 (linear) from 4 grid(s)...


/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dat

  dp: smooth merge:   0%|          | 0/1302 [00:00<?, ?it/s]

Saved /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/dp1_500m.nc
  -> /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/dp1_500m.nc
Smooth merge dp2 (linear) from 4 grid(s)...


/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dat

...

  spr: smooth merge:   0%|          | 0/1302 [00:00<?, ?it/s]

Saved /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/spr2_500m.nc
  -> /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/spr2_500m.nc
Smooth merge spr3 (linear) from 4 grid(s)...


/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={"time": 10_000})
/lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/utils/postprocessing_binwaves_bmus.py:624: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dat

  spr: smooth merge:   0%|          | 0/1302 [00:00<?, ?it/s]

Saved /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/spr3_500m.nc
  -> /nfs/home/geocean/montanoj/New_data_Shoreshop/outputs/merged_500m_binwaves_bmus/spr3_500m.nc
Merged partition files:
  dp1      dp1_500m.nc  (1335.8 MB, linear blend)
  dp2      dp2_500m.nc  (1376.6 MB, linear blend)
  dp3      dp3_500m.nc  (1364.1 MB, linear blend)
  phs1     phs1_500m.nc  (1485.0 MB, linear blend)
  phs2     phs2_500m.nc  (1532.9 MB, linear blend)
  phs3     phs3_500m.nc  (1555.0 MB, linear blend)
  ptp1     ptp1_500m.nc  (1275.9 MB, linear blend)
  ptp2     ptp2_500m.nc  (1363.6 MB, linear blend)
  ptp3     ptp3_500m.nc  (1360.2 MB, linear blend)
  spr1     spr1_500m.nc  (1466.0 MB, linear blend)
  spr2     spr2_500m.nc  (1499.6 MB, linear blend)
  spr3     spr3_500m.nc  (1492.2 MB, linear blend)


## 4. Generate webpage GeoJSON

In [3]:
geojson_paths = pb.generate_webpage_geojson(
    merged_dir=MERGED_DIR,
    output_dir=WEBPAGE_DIR,
    merged_suffix="_500m",
)

for key, path in geojson_paths.items():
    size_mb = path.stat().st_size / 1e6
    print(f"{key}: {path} ({size_mb:.1f} MB)")

wave_statistics_all:   0%|          | 0/1269 [00:00<?, ?it/s]

wave_statistics_hs:   0%|          | 0/1269 [00:00<?, ?it/s]

wave_statistics_dp:   0%|          | 0/1269 [00:00<?, ?it/s]

all: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/webpage_binwaves_bmus_500m/wave_statistics_all.geojson (0.8 MB)
hs: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/webpage_binwaves_bmus_500m/wave_statistics_hs.geojson (1.2 MB)
dp: /lustre/geocean/WORK/users/montanoj/personal/Wind_Metamodel/webpage_binwaves_bmus_500m/wave_statistics_dp.geojson (1.2 MB)
